# ML Zoomcamp 2026 — Module 3 Homework [DRAFT]

> This is a draft carried over from the 2025 cohort. Questions/dataset/models may still change
> before the module opens — re-check `../../cohorts/2026/03-classification/homework.md` before
> submitting.
>
> Sometimes your answer won't match one of the options exactly. That's fine — pick the closest
> option, and the higher one if you're exactly in between two.

Dataset: [Lead scoring dataset](https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv)

Target for classification: `converted` — whether the client signed up to the platform.


## Setup


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LogisticRegression

%matplotlib inline


## Data preparation

* Check if missing values are present in the features.
* If there are missing values:
  * For categorical features, replace them with `'NA'`.
  * For numerical features, replace them with `0.0`.


In [2]:
url = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'
df = pd.read_csv(url)
df.head()


In [4]:
# Check for missing values
df.isnull().sum()


lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64


In [8]:
df.dtypes

categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']


In [11]:
# Fill missing values: 'NA' for categorical, 0.0 for numerical
for c in categorical:
    df[c] = df[c].fillna('NA')

for n in numerical:
    df[n] = df[n].fillna(0)



## Question 1. Column with missing values

What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology`
- `healthcare`
- `retail`


In [13]:
# Mode of the industry column
df.industry.value_counts() # retail


industry
retail           203
finance          200
other            198
healthcare       187
education        187
technology       179
manufacturing    174
NA               134
Name: count, dtype: int64


## Question 2. Correlation matrix

Create the correlation matrix for the numerical features of the dataset.

What are the two features that have the biggest correlation? (Only consider the pairs below.)

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`


In [39]:
# Correlation matrix of numerical features
corr_dict = {}
for i in range(len(numerical)-1):
    for j in range(i+1, len(numerical)):
        key = 'corr_' + numerical[i] + '_' + numerical[j]
        corr_dict[key] = float(df[numerical[i]].corr(df[numerical[j]]))

df[numerical].corr()


## Split the data

* Split into train/val/test with a 60%/20%/20% distribution.
* Use `train_test_split` from Scikit-Learn, seed `42`.
* Make sure `converted` is not in the feature DataFrames.


In [57]:
# Two-step split into train/val/test (60/20/20), seed=42, drop converted from features

from sklearn.model_selection import train_test_split

X_full_train, X_test = train_test_split(df, test_size=0.2, random_state=42)
X_train, X_val = train_test_split(X_full_train, test_size=0.2, random_state=42)

X_test = X_test.reset_index(drop=True)
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_full_train = X_full_train.reset_index(drop=True)

y_test = X_test.converted
y_train = X_train.converted
y_val = X_val.converted
y_full_train = X_full_train.converted

del X_test['converted']
del X_train['converted']
del X_val['converted']
del X_full_train['converted']


## Question 3. Mutual information

* Calculate the mutual information score between `converted` and the other categorical variables,
  using the training set only.
* Round the scores to 2 decimals with `round(score, 2)`.

Which variable has the biggest mutual information score?

- `industry`
- `location`
- `lead_source`
- `employment_status`


In [45]:
# Mutual information score per categorical variable, rounded to 2 decimals
from sklearn.metrics import mutual_info_score

mutual_info_dict = {}

for c in categorical:
    mutual_info_dict[c] = mutual_info_score(df[c], df['converted'])

mutual_info_dict


{'lead_source': 0.026573987738060995,
 'industry': 0.007267435279688886,
 'employment_status': 0.011069560968622674,
 'location': 0.0014269064526338665}


## Question 4. Logistic regression

* Include categorical variables using one-hot encoding.
* Fit on the training dataset with:
  `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
* Calculate accuracy on the validation dataset, rounded to 2 decimals.

What accuracy did you get?

- 0.64
- 0.74
- 0.84
- 0.94


In [62]:
# One-hot encode, train LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)
train_dict = df[categorical+numerical].to_dict(orient='records')

dv.fit(train_dict)

X_train_dict = X_train.to_dict(orient='records')
X_train_final = dv.transform(X_train_dict)

X_val_dict = X_val.to_dict(orient='records')
X_val_final = dv.transform(X_val_dict)


In [84]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
repr(model.fit(X_train_final, y_train))

model.intercept_, model.coef_

y_val_predict = model.predict(X_val_final)
y_val_predict_proba = model.predict_proba(X_val_final)[:, 1]


array([0.61283518, 0.79485653, 0.52375878, 0.47029634, 0.56627331,
       0.4353565 , 0.86656502, 0.84265411, 0.83064054, 0.61371712,
       0.54454813, 0.77302173, 0.68475907, 0.76346789, 0.51785968,
       0.91359667, 0.52485964, 0.41580329, 0.29986275, 0.84666592,
       0.78811556, 0.73058919, 0.4367615 , 0.6414975 , 0.41163979,
       0.75028314, 0.89796124, 0.33529298, 0.42480806, 0.96800582,
       0.91919136, 0.37126582, 0.65048494, 0.90272061, 0.74600835,
       0.63602409, 0.81779166, 0.83046243, 0.65209774, 0.30757423,
       0.7869325 , 0.35319064, 0.964229  , 0.62929742, 0.51169483,
       0.53120288, 0.81883429, 0.74077945, 0.72533031, 0.68320355,
       0.4659718 , 0.84123492, 0.55625639, 0.92558517, 0.6462457 ,
       0.61451005, 0.63114837, 0.2792807 , 0.47956643, 0.57026858,
       0.35428028, 0.61399379, 0.38384637, 0.60516856, 0.84883422,
       0.74964404, 0.88945506, 0.71778283, 0.95241276, 0.88979821,
       0.75132619, 0.33486261, 0.61099778, 0.51121072, 0.63218

In [108]:
# Validation accuracy, rounded to 2 decimals
accuracy = (y_val == y_val_predict).mean().round(2)

tp = ((y_val == y_val_predict) & (y_val_predict == 1)).sum()
fp = ((y_val_predict == 1) & (y_val == 0)).sum()
fn = ((y_val_predict == 0) & (y_val == 1)).sum()

# Precision = TP / (TP+FP)
# Recall = TP / (TP+FN)
precision = (tp / (tp + fp)).round(2)
recall = (tp / (tp + fn)).round(2)

accuracy, precision, recall


(np.float64(0.71), np.float64(0.66), np.float64(0.95))


In [109]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

print('Accuracy: ' + str(accuracy.round(2) == round(accuracy_score(y_val, y_val_predict), 2)))
print('Precision: ' + str(precision.round(2) == round(precision_score(y_val, y_val_predict), 2)))
print('Recall: ' + str(recall == round(recall_score(y_val, y_val_predict), 2)))


Accuracy: True
Precision: True
Recall: True


## Question 5. Feature elimination

* Train a model using the same features/params as Q4 (without rounding) — this is the baseline
  accuracy.
* Exclude each feature one at a time, retrain, and record the accuracy.
* For each feature, compute `baseline_accuracy - accuracy_without_feature`.

Which of the following has the smallest difference (not necessarily positive)?

- `'industry'`
- `'employment_status'`
- `'lead_score'`


In [117]:
# Baseline accuracy (unrounded), then leave-one-feature-out accuracy differences
def train_logistic_regression(X_train, y_train, features):
    X_train_dict = X_train[features].to_dict(orient='records')
    X_train_final = dv.transform(X_train_dict)

    model.fit(X_train_final, y_train)

def get_accuracy_score(X_test, y_test, features):
    X_test_dict = X_test[features].to_dict(orient='records')
    X_test_final = dv.transform(X_test_dict)
    y_pred = model.predict(X_test_final)
    return (y_pred == y_test).mean()

features = categorical + numerical

accuracy_scores_dict = {}

train_logistic_regression(X_full_train, y_full_train, features)
base_accuracy = get_accuracy_score(X_test, y_test, features)

for i in range(len(features)):
    curr_features = features.copy()
    removed = curr_features.pop(i)
    train_logistic_regression(X_full_train, y_full_train, curr_features)
    accuracy = get_accuracy_score(X_test, y_test, curr_features)
    accuracy_scores_dict['Removed ' + removed] = base_accuracy - accuracy
    
accuracy_scores_dict


{'Removed lead_source': np.float64(0.010238907849829282),
 'Removed industry': np.float64(0.0),
 'Removed employment_status': np.float64(0.010238907849829282),
 'Removed location': np.float64(0.0),
 'Removed number_of_courses_viewed': np.float64(0.06825938566552903),
 'Removed annual_income': np.float64(-0.08191126279863481),
 'Removed interaction_count': np.float64(0.027303754266211566),
 'Removed lead_score': np.float64(-0.0034129692832765013)}


## Question 6. Regularized logistic regression

* Try `C` in `[0.01, 0.1, 1, 10, 100]`.
* Train using all the features as in Q4.
* Calculate validation accuracy, rounded to 3 decimals.

Which `C` gives the best validation accuracy? (If tied, pick the smallest `C`.)

- 0.01
- 0.1
- 1
- 10
- 100


In [127]:
# Loop over C values, train, print validation accuracy rounded to 3 decimals
c_values = [0.00001, 0.0001, 0.001 ,0.01, 0.1, 1, 10, 100, 1000]

def train_logistic_regression_regularised(X_train, y_train, features, r):
    model = LogisticRegression(solver='liblinear', C=r, max_iter=1000, random_state=42)
    
    X_train_dict = X_train[features].to_dict(orient='records')
    X_train_final = dv.transform(X_train_dict)

    model.fit(X_train_final, y_train)
    return model

def get_accuracy_score(X_test, y_test, features, model):
    X_test_dict = X_test[features].to_dict(orient='records')
    X_test_final = dv.transform(X_test_dict)
    y_pred = model.predict(X_test_final)
    return (y_pred == y_test).mean()

reg_dict = {}

for r in c_values:
    reg_model = train_logistic_regression_regularised(X_full_train, y_full_train, features, r)
    reg_dict[r] = get_accuracy_score(X_test, y_test, features, reg_model)

reg_dict


{1e-05: np.float64(0.6655290102389079),
 0.0001: np.float64(0.6655290102389079),
 0.001: np.float64(0.6689419795221843),
 0.01: np.float64(0.7337883959044369),
 0.1: np.float64(0.7372013651877133),
 1: np.float64(0.7372013651877133),
 10: np.float64(0.7372013651877133),
 100: np.float64(0.7372013651877133),
 1000: np.float64(0.7372013651877133)}


## Submit the results

* Submit here: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw03
* If your answer doesn't match exactly, pick the closest option (higher one if exactly in between).
